# Build the JEM validation library with SEEA-Sim, in Google Colab

Makes **`JEM_cycle1.ssim`** from *SEEA-Sim JEM validation 2015-2022* with the SEEA-Sim tables already inside, and saves it to Drive:

1. installs SyncroSim (Linux), ST-Sim and **SEEA-Sim 1.0.3** (built from the source embedded below);
2. builds the ST-Sim library with your `build_and_test.py` (full set-up, no run);
3. adds the SEEA-Sim tables (`4_seeasim/`) with `add_seeasim.py`: package, SEEA stock and flow groups, services, accounting periods, transition classes, carbon roles, supply rules, prices, asset valuation, the extra ST-Sim output, and the pipeline (1 ST-Sim, 2 SEEA-Sim Accounts);
4. reads the library back and lists every SEEA-Sim table, then copies the library to Drive (`colab_build_<date>/`).

The run itself happens on the laptop, because SyncroSim on Linux cannot write rasters (GDAL `FlushCache` mismatch). Install `stsimseea-1-0-3.ssimpkg` in Studio, open the library, and run.

Run top to bottom. Cells 2 to 4 take about 10 minutes, every new Colab session. Mount the Google account that owns **Thesis**.

In [ ]:
# 1. Mount Drive, find the folder, copy it to the Colab disk (SQLite libraries can break on the Drive mount)
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
FOLDER_NAME = 'SEEA-Sim JEM validation 2015-2022'
hits = glob.glob(f'/content/drive/MyDrive/**/{FOLDER_NAME}', recursive=True)
print(hits)
assert hits, "Folder not found. If it is in 'Shared with me', add a shortcut to My Drive first, or set DRIVE_DIR by hand."
DRIVE_DIR = hits[0]
WORK = '/content/jem_validation'
shutil.copytree(DRIVE_DIR, WORK, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('colab_build_*', '*.ssim', '*.ssim.*'))
print(sorted(os.listdir(WORK)))

In [ ]:
%%bash
# 2. System libraries and Mono (the .NET runtime SyncroSim uses on Linux)
set -e
apt-get -qq update
apt-get -qq install -y ca-certificates gnupg unzip > /dev/null
gpg --homedir /tmp --no-default-keyring --keyring /usr/share/keyrings/mono-official-archive-keyring.gpg \
    --keyserver hkp://keyserver.ubuntu.com:80 --recv-keys 3FA7E0328081BFF6A14DA29AA6A19B38D3D831EF
CODENAME=$(. /etc/os-release && echo $VERSION_CODENAME)
echo "deb [signed-by=/usr/share/keyrings/mono-official-archive-keyring.gpg] https://download.mono-project.com/repo/ubuntu stable-$CODENAME main" \
    > /etc/apt/sources.list.d/mono-official-stable.list
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mono-complete libgdal-dev libproj-dev libgeos-dev \
    libsqlite3-dev libspatialite-dev libudunits2-dev gdal-bin proj-bin > /dev/null
mono --version | head -1

In [ ]:
%%bash
# 3. Miniconda, only to get the GDAL C# bindings SyncroSim needs for rasters
set -e
if [ ! -d /root/miniconda3 ]; then
  curl -fsSL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/mc.sh
  bash /tmp/mc.sh -b -p /root/miniconda3 > /dev/null
fi
/root/miniconda3/bin/conda config --add channels conda-forge
/root/miniconda3/bin/conda config --set channel_priority strict
/root/miniconda3/bin/conda create -y -q --override-channels -c conda-forge --name gdal-mono gdal-csharp=1.1.1 > /dev/null
ls /root/miniconda3/envs/gdal-mono/lib/*_csharp.dll

In [ ]:
%%bash
# 4. SyncroSim itself, plus ST-Sim
set -e
SSIM_VERSION=3-1-29          # change if syncrosim.com lists a newer Linux build
SS=/root/syncrosim
if [ ! -f $SS/SyncroSim.Console.exe ]; then
  curl -fsSL "https://downloads.syncrosim.com/$SSIM_VERSION/syncrosim-linux-$SSIM_VERSION.zip" -o /tmp/ss.zip
  unzip -q /tmp/ss.zip -d $SS
fi
chmod 755 $SS/SyncroSim.Console.exe $SS/SyncroSim.PackageManager.exe
cp /root/miniconda3/envs/gdal-mono/lib/*_csharp.dll $SS/
cp /root/miniconda3/envs/gdal-mono/lib/*.so* $SS/
# drop conda copies that clash with the system libraries
rm -f $SS/libsqlite3.so* $SS/libproj.so* $SS/libgeos*.so* $SS/libgdal.so* $SS/libspatialite.so* $SS/libcurl.so*
[ -e /lib/x86_64-linux-gnu/libspatialite.so.7 ] || ln -s /lib/x86_64-linux-gnu/libspatialite.so.8 /lib/x86_64-linux-gnu/libspatialite.so.7 || true
ldconfig
printf '#!/bin/bash\nexec mono /root/syncrosim/SyncroSim.Console.exe "$@"\n' > /usr/local/bin/ssim
printf '#!/bin/bash\nexec mono /root/syncrosim/SyncroSim.PackageManager.exe "$@"\n' > /usr/local/bin/ssimpm
chmod +x /usr/local/bin/ssim /usr/local/bin/ssimpm
export LD_LIBRARY_PATH=$SS:${LD_LIBRARY_PATH:-}
ssim --conda --path=/root/miniconda3
ssimpm --install=stsim
ssimpm --install=stsimsf || echo "stsimsf not installed (fine if this ST-Sim already includes stocks and flows)"
ssimpm --list --installed

In [ ]:
# 5. Python side
!pip -q install pysyncrosim rasterio
import os
os.environ['LD_LIBRARY_PATH'] = '/root/syncrosim:' + os.environ.get('LD_LIBRARY_PATH', '')
import pysyncrosim as ps
s = ps.Session(location='/root/syncrosim')
print('SyncroSim', s.version())

In [ ]:
# 6a. SEEA-Sim 1.0.3 source (same as src/ in github.com/Shlhnj/stsimseea)
PKG_FILES = {
 "package.xml": "<?xml version=\"1.0\" encoding=\"utf-8\"?>\n<!--\n  SEEA-Sim (package name: stsimseea)\n  SyncroSim add-on that turns ST-Sim 4.5 output (state classes, transitions,\n  stocks and flows) into SEEA Ecosystem Accounting tables.\n\n  Syntax follows ApexRMS/stsimecodep (builtFor, no <dataSheets> wrapper, no\n  explicit ID/ScenarioId columns) and ApexRMS/helloworldPy (Python transformer\n  with condaEnv). ST-Sim datasheet and column names checked against\n  ApexRMS/stsim src/package.xml, version 4.5.4.\n\n  Run it as a second pipeline stage after stsim_Runtime.\n-->\n<package name=\"stsimseea\" displayName=\"SEEA-Sim\" description=\"SEEA Ecosystem Accounting tables from ST-Sim output\" version=\"1.0.3\" minSyncroSimVersion=\"3.1.14\" url=\"https://github.com/Shlhnj/stsimseea\">\n\n  <builtFor package=\"stsim\" version=\"4.5.4\"/>\n\n  <!-- ===================== Project definitions ===================== -->\n\n  <dataSheet name=\"Service\" displayName=\"Ecosystem Service\" dataScope=\"Project\" displayMember=\"Name\">\n    <column name=\"Name\" dataType=\"String\"/>\n    <column name=\"ServiceType\" displayName=\"Type\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Provisioning|2:Regulating|3:Cultural\"/>\n    <column name=\"Units\" displayName=\"Physical Units\" dataType=\"String\" isOptional=\"True\"/>\n    <column name=\"Description\" dataType=\"String\" isOptional=\"True\"/>\n  </dataSheet>\n\n  <!-- ===================== Scenario inputs ===================== -->\n\n  <dataSheet name=\"Options\" displayName=\"Options\" isSingleRow=\"True\">\n    <column name=\"ReportByStratum\" displayName=\"Report by primary stratum (accounting area)\" dataType=\"Boolean\"/>\n    <column name=\"Currency\" dataType=\"String\" isOptional=\"True\"/>\n  </dataSheet>\n\n  <dataSheet name=\"AccountingPeriod\" displayName=\"Accounting Periods\">\n    <column name=\"OpeningTimestep\" displayName=\"Opening\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"GreaterEqual\" formula1=\"0\" format=\"d\" allowDbNull=\"False\"/>\n    <column name=\"ClosingTimestep\" displayName=\"Closing\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"GreaterEqual\" formula1=\"0\" format=\"d\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"OpeningTimestep|ClosingTimestep\"/>\n  </dataSheet>\n\n  <dataSheet name=\"TransitionTypeClass\" displayName=\"Transition Type Classification\">\n    <column name=\"TransitionTypeId\" displayName=\"Transition Type\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_TransitionType\" allowDbNull=\"False\"/>\n    <column name=\"ChangeType\" displayName=\"Change Type\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Managed|2:Unmanaged|3:Catastrophic\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"TransitionTypeId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"CarbonStockGroup\" displayName=\"Carbon Stock Groups\">\n    <column name=\"StockGroupId\" displayName=\"Stock Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\" allowDbNull=\"False\"/>\n    <column name=\"Role\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Ecosystem pool|2:Atmosphere|3:Water|4:Long-term burial|5:Other external\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"StockGroupId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"CarbonFlowGroup\" displayName=\"Carbon Flow Groups\">\n    <column name=\"FlowGroupId\" displayName=\"Flow Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_FlowGroup\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"FlowGroupId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ServiceSupply\" displayName=\"Service Supply Rules\">\n    <column name=\"ServiceId\" displayName=\"Service\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\" allowDbNull=\"False\"/>\n    <column name=\"StratumId\" displayName=\"Stratum\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" isOptional=\"True\"/>\n    <column name=\"Basis\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Area|2:Stock group|3:Flow group\" allowDbNull=\"False\"/>\n    <column name=\"StockGroupId\" displayName=\"Stock Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\" isOptional=\"True\"/>\n    <column name=\"FlowGroupId\" displayName=\"Flow Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_FlowGroup\" isOptional=\"True\"/>\n    <column name=\"Coefficient\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ServicePrice\" displayName=\"Service Prices\" hasTimestep=\"True\">\n    <column name=\"ServiceId\" displayName=\"Service\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\" allowDbNull=\"False\"/>\n    <column name=\"Price\" displayName=\"Price per physical unit\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"AssetValuation\" displayName=\"Asset Valuation\">\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" isOptional=\"True\"/>\n    <column name=\"DiscountRate\" displayName=\"Discount Rate\" dataType=\"Double\" validationType=\"Decimal\" validationCondition=\"GreaterEqual\" formula1=\"0.0\" allowDbNull=\"False\"/>\n    <column name=\"AssetLifeYears\" displayName=\"Asset Life (years)\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"Greater\" formula1=\"0\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ConditionChange\" displayName=\"Condition Change\">\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" allowDbNull=\"False\"/>\n    <column name=\"OpeningTimestep\" displayName=\"Opening\" dataType=\"Integer\" format=\"d\" isOptional=\"True\"/>\n    <column name=\"ClosingTimestep\" displayName=\"Closing\" dataType=\"Integer\" format=\"d\" isOptional=\"True\"/>\n    <column name=\"Change\" displayName=\"Change in condition index\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <!-- ===================== Outputs ===================== -->\n  <!-- Sign convention: additions positive, reductions negative, so each\n       account's Net change equals the sum of its change entries. -->\n\n  <dataSheet name=\"OutputExtent\" displayName=\"Extent Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputChangeMatrix\" displayName=\"Ecosystem Type Change Matrix\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"FromStateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ToStateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCarbonStock\" displayName=\"Carbon Stock Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"StockGroupId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCarbonFlow\" displayName=\"Carbon Flow Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"TransitionTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_TransitionType\" isOptional=\"True\"/>\n    <column name=\"FromStockTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockType\"/>\n    <column name=\"ToStockTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockType\"/>\n    <column name=\"Category\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputServiceSupply\" displayName=\"Physical Supply\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ServiceId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputServiceValue\" displayName=\"Monetary Supply\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ServiceId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputAssetAccount\" displayName=\"Monetary Asset Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCheck\" displayName=\"Reconciliation Checks\" hasIteration=\"True\">\n    <column name=\"Account\" dataType=\"String\"/>\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"Item\" dataType=\"String\"/>\n    <column name=\"Residual\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <!-- ===================== Transformer ===================== -->\n\n  <transformer\n    name=\"Main\"\n    displayName=\"SEEA-Sim Accounts\"\n    isRunnable=\"True\"\n    programName=\"python\"\n    programArguments=\"transformer.py\"\n    condaEnv=\"stsimseea.yml\"\n    condaEnvVersion=\"2\">\n      <dataSheet name=\"Options\" type=\"Input\"/>\n      <dataSheet name=\"AccountingPeriod\" type=\"Input\"/>\n      <dataSheet name=\"TransitionTypeClass\" type=\"Input\"/>\n      <dataSheet name=\"CarbonStockGroup\" type=\"Input\"/>\n      <dataSheet name=\"CarbonFlowGroup\" type=\"Input\"/>\n      <dataSheet name=\"ServiceSupply\" type=\"Input\"/>\n      <dataSheet name=\"ServicePrice\" type=\"Input\"/>\n      <dataSheet name=\"AssetValuation\" type=\"Input\"/>\n      <dataSheet name=\"ConditionChange\" type=\"Input\"/>\n      <dataSheet name=\"OutputExtent\" type=\"Output\"/>\n      <dataSheet name=\"OutputChangeMatrix\" type=\"Output\"/>\n      <dataSheet name=\"OutputCarbonStock\" type=\"Output\"/>\n      <dataSheet name=\"OutputCarbonFlow\" type=\"Output\"/>\n      <dataSheet name=\"OutputServiceSupply\" type=\"Output\"/>\n      <dataSheet name=\"OutputServiceValue\" type=\"Output\"/>\n      <dataSheet name=\"OutputAssetAccount\" type=\"Output\"/>\n      <dataSheet name=\"OutputCheck\" type=\"Output\"/>\n  </transformer>\n\n  <!-- ===================== Layouts ===================== -->\n\n  <layout type=\"Project\">\n    <item name=\"Service\"/>\n  </layout>\n\n  <layout type=\"Scenario\">\n    <group name=\"SEEASim\" displayName=\"SEEA-Sim\">\n      <item name=\"Options\"/>\n      <item name=\"AccountingPeriod\"/>\n      <item name=\"TransitionTypeClass\"/>\n      <group name=\"Carbon\">\n        <item name=\"CarbonStockGroup\"/>\n        <item name=\"CarbonFlowGroup\"/>\n      </group>\n      <group name=\"Services\" displayName=\"Ecosystem Services\">\n        <item name=\"ServiceSupply\"/>\n        <item name=\"ServicePrice\"/>\n        <item name=\"AssetValuation\"/>\n        <item name=\"ConditionChange\"/>\n      </group>\n      <group name=\"Results\">\n        <item name=\"OutputExtent\"/>\n        <item name=\"OutputChangeMatrix\"/>\n        <item name=\"OutputCarbonStock\"/>\n        <item name=\"OutputCarbonFlow\"/>\n        <item name=\"OutputServiceSupply\"/>\n        <item name=\"OutputServiceValue\"/>\n        <item name=\"OutputAssetAccount\"/>\n        <item name=\"OutputCheck\"/>\n      </group>\n    </group>\n  </layout>\n\n</package>\n",
 "seea_accounts.py": "\"\"\"\nSEEA-Sim accounting core.\n\nPure pandas. No SyncroSim import, so it can be tested anywhere. The\ntransformer (transformer.py) reads ST-Sim datasheets, passes them here as\nDataFrames and writes the results back.\n\nInputs use ST-Sim 4.5 output column names:\n  stsim_OutputStratumState          Iteration, Timestep, StratumId, StateClassId, Amount\n  stsim_OutputStratumTransitionState Iteration, Timestep, StratumId, TransitionTypeId,\n                                    StateClassId, EndStateClassId, Amount\n  stsim_OutputStock                 Iteration, Timestep, StratumId, StateClassId,\n                                    StockGroupId, Amount\n  stsim_OutputFlow                  Iteration, Timestep, FromStratumId, FromStateClassId,\n                                    FromStockTypeId, ToStockTypeId, TransitionTypeId,\n                                    FlowGroupId, Amount\nId columns can hold names or integers; values are only used as keys.\n\nConventions\n  * A period (o, c) covers the transitions and flows of timesteps o+1 .. c.\n    Opening = state at the end of timestep o, closing = state at the end of c.\n  * Additions are positive and reductions negative, so Net change equals the\n    sum of the change entries in every account.\n  * Every account is computed per iteration. SyncroSim summarises across\n    iterations.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nALL = \"__all__\"  # placeholder stratum when results are not split by stratum\n\nMANAGED, UNMANAGED, CATASTROPHIC = \"Managed\", \"Unmanaged\", \"Catastrophic\"\nCHANGE_TYPES = {1: MANAGED, 2: UNMANAGED, 3: CATASTROPHIC}\n\nPOOL, ATMOSPHERE, WATER, BURIAL, OTHER = (\n    \"Ecosystem pool\", \"Atmosphere\", \"Water\", \"Long-term burial\", \"Other external\")\nROLES = {1: POOL, 2: ATMOSPHERE, 3: WATER, 4: BURIAL, 5: OTHER}\n\nAREA, STOCK, FLOW = \"Area\", \"Stock group\", \"Flow group\"\nBASES = {1: AREA, 2: STOCK, 3: FLOW}\n\nPERIOD = [\"OpeningTimestep\", \"ClosingTimestep\"]\n\n\nclass AccountError(ValueError):\n    \"\"\"Input problem the user has to fix; the message says how.\"\"\"\n\n\ndef _log(log, msg):\n    (log or print)(msg)\n\n\ndef list_value(v, mapping: dict, what: str):\n    \"\"\"SyncroSim List columns may arrive as 1, '1' or the label.\"\"\"\n    if pd.isna(v):\n        raise AccountError(f\"{what} is empty.\")\n    labels = {s.lower(): s for s in mapping.values()}\n    if isinstance(v, str) and v.strip().lower() in labels:\n        return labels[v.strip().lower()]\n    try:\n        return mapping[int(float(v))]\n    except (ValueError, KeyError):\n        raise AccountError(f\"Unknown {what}: {v!r}. Allowed: {list(mapping.values())}\")\n\n\ndef annuity_factor(rate: float, life: int) -> float:\n    \"\"\"Present value of 1 per year for `life` years at `rate`.\"\"\"\n    if rate == 0:\n        return float(life)\n    return (1 - (1 + rate) ** -life) / rate\n\n\ndef _need(df: pd.DataFrame, cols, name: str):\n    missing = [c for c in cols if c not in df.columns]\n    if missing:\n        raise AccountError(f\"{name} is missing column(s) {missing}; found {list(df.columns)}.\")\n\n\ndef _stratum(df: pd.DataFrame, col: str, by_stratum: bool) -> pd.DataFrame:\n    df = df.copy()\n    df[\"StratumId\"] = df[col] if by_stratum else ALL\n    return df\n\n\ndef _finish(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Replace the placeholder stratum with null before saving.\"\"\"\n    if \"StratumId\" in df.columns:\n        df[\"StratumId\"] = df[\"StratumId\"].astype(object).where(df[\"StratumId\"] != ALL, None)\n    return df.reset_index(drop=True)\n\n\ndef default_periods(state: pd.DataFrame) -> list[tuple[int, int]]:\n    return [(int(state[\"Timestep\"].min()), int(state[\"Timestep\"].max()))]\n\n\ndef _check_timesteps(df: pd.DataFrame, timesteps, name: str):\n    have = set(df[\"Timestep\"].unique())\n    missing = sorted(t for t in timesteps if t not in have)\n    if missing:\n        raise AccountError(\n            f\"{name} has no output for timestep(s) {missing}. Set its summary output \"\n            f\"frequency so these timesteps are written (every timestep is safest).\")\n\n\n# ---------------------------------------------------------------- extent\n\nEXTENT_ENTRIES = [\n    \"Opening extent\",\n    \"Additions: managed expansion\",\n    \"Additions: unmanaged expansion\",\n    \"Reductions: managed reduction\",\n    \"Reductions: unmanaged reduction\",\n    \"Net change in extent\",\n    \"Closing extent\",\n]\n\n\ndef _prep_state(state, by_stratum):\n    _need(state, [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"Amount\"],\n          \"stsim_OutputStratumState\")\n    df = _stratum(state, \"StratumId\", by_stratum)\n    return df.groupby([\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\"],\n                      as_index=False)[\"Amount\"].sum()\n\n\ndef _prep_trans(trans, tt_class: dict, by_stratum, log):\n    _need(trans, [\"Iteration\", \"Timestep\", \"StratumId\", \"TransitionTypeId\",\n                  \"StateClassId\", \"EndStateClassId\", \"Amount\"],\n          \"stsim_OutputStratumTransitionState\")\n    df = _stratum(trans, \"StratumId\", by_stratum)\n    df = df[df[\"StateClassId\"] != df[\"EndStateClassId\"]].copy()\n    unlisted = sorted({str(t) for t in df[\"TransitionTypeId\"].unique() if t not in tt_class})\n    if unlisted:\n        _log(log, \"SEEA-Sim: transition types not in Transition Type Classification are \"\n                  f\"treated as Unmanaged: {', '.join(unlisted)}\")\n    df[\"ChangeType\"] = df[\"TransitionTypeId\"].map(tt_class).fillna(UNMANAGED)\n    return df\n\n\ndef extent_components(state, trans, periods, tt_class, by_stratum=False, log=None):\n    \"\"\"\n    One row per (Iteration, StratumId, period, StateClassId) with\n    Opening, Closing, Add_<type>, Red_<type> (positive areas) and Residual\n    (= closing - opening - additions + reductions; zero unless something\n    other than recorded transitions changed the class areas).\n    \"\"\"\n    st = _prep_state(state, by_stratum)\n    tr = _prep_trans(trans, tt_class, by_stratum, log)\n    keys = [\"Iteration\", \"StratumId\"]\n    out = []\n    for o, c in periods:\n        _check_timesteps(st, [o, c], \"stsim_OutputStratumState\")\n        op = st[st.Timestep == o].drop(columns=\"Timestep\").rename(columns={\"Amount\": \"Opening\"})\n        cl = st[st.Timestep == c].drop(columns=\"Timestep\").rename(columns={\"Amount\": \"Closing\"})\n        t = tr[(tr.Timestep > o) & (tr.Timestep <= c)]\n        add = (t.groupby(keys + [\"EndStateClassId\", \"ChangeType\"])[\"Amount\"].sum()\n                .unstack(\"ChangeType\").add_prefix(\"Add_\").reset_index()\n                .rename(columns={\"EndStateClassId\": \"StateClassId\"}))\n        red = (t.groupby(keys + [\"StateClassId\", \"ChangeType\"])[\"Amount\"].sum()\n                .unstack(\"ChangeType\").add_prefix(\"Red_\").reset_index())\n        df = op.merge(cl, on=keys + [\"StateClassId\"], how=\"outer\")\n        for part in (add, red):\n            if not part.empty:\n                df = df.merge(part, on=keys + [\"StateClassId\"], how=\"outer\")\n        for p in (\"Add_\", \"Red_\"):\n            for ct in (MANAGED, UNMANAGED, CATASTROPHIC):\n                if p + ct not in df.columns:\n                    df[p + ct] = 0.0\n        df = df.fillna({c_: 0.0 for c_ in df.columns if c_ not in keys + [\"StateClassId\"]})\n        df[\"OpeningTimestep\"], df[\"ClosingTimestep\"] = o, c\n        out.append(df)\n    comp = pd.concat(out, ignore_index=True)\n    adds = comp[[f\"Add_{t}\" for t in (MANAGED, UNMANAGED, CATASTROPHIC)]].sum(axis=1)\n    reds = comp[[f\"Red_{t}\" for t in (MANAGED, UNMANAGED, CATASTROPHIC)]].sum(axis=1)\n    comp[\"Residual\"] = comp[\"Closing\"] - comp[\"Opening\"] - adds + reds\n    return comp\n\n\ndef extent_account(comp: pd.DataFrame) -> pd.DataFrame:\n    vals = {\n        \"Opening extent\": comp[\"Opening\"],\n        \"Additions: managed expansion\": comp[\"Add_Managed\"],\n        \"Additions: unmanaged expansion\": comp[\"Add_Unmanaged\"] + comp[\"Add_Catastrophic\"],\n        \"Reductions: managed reduction\": -comp[\"Red_Managed\"],\n        \"Reductions: unmanaged reduction\": -(comp[\"Red_Unmanaged\"] + comp[\"Red_Catastrophic\"]),\n        \"Net change in extent\": comp[\"Closing\"] - comp[\"Opening\"],\n        \"Closing extent\": comp[\"Closing\"],\n    }\n    base = comp[[\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\"]]\n    rows = [base.assign(Entry=e, Amount=vals[e].values) for e in EXTENT_ENTRIES]\n    return _finish(pd.concat(rows, ignore_index=True))\n\n\ndef change_matrix(state, trans, periods, tt_class, by_stratum=False, log=None):\n    \"\"\"\n    From x To areas per period. Off-diagonal cells are gross transitions\n    summed over the period's timesteps; the diagonal is opening area minus\n    everything that left the class. Row totals equal opening extent and\n    column totals equal closing extent. A cell that changes more than once\n    in a period (A to B to A) is counted in every move, so off-diagonals are\n    gross flows, not a map-to-map comparison, and the diagonal understates\n    the area that never changed.\n    \"\"\"\n    st = _prep_state(state, by_stratum)\n    tr = _prep_trans(trans, tt_class, by_stratum, log=lambda m: None)\n    keys = [\"Iteration\", \"StratumId\"]\n    out, checks = [], []\n    for o, c in periods:\n        t = tr[(tr.Timestep > o) & (tr.Timestep <= c)]\n        off = (t.groupby(keys + [\"StateClassId\", \"EndStateClassId\"], as_index=False)[\"Amount\"].sum()\n                .rename(columns={\"StateClassId\": \"FromStateClassId\",\n                                 \"EndStateClassId\": \"ToStateClassId\"}))\n        op = st[st.Timestep == o].drop(columns=\"Timestep\")\n        left = off.groupby(keys + [\"FromStateClassId\"], as_index=False)[\"Amount\"].sum()\n        left = left.rename(columns={\"FromStateClassId\": \"StateClassId\", \"Amount\": \"Left\"})\n        diag = op.merge(left, on=keys + [\"StateClassId\"], how=\"left\").fillna({\"Left\": 0.0})\n        diag = diag.assign(FromStateClassId=diag.StateClassId, ToStateClassId=diag.StateClassId,\n                           Amount=diag.Amount - diag.Left)\n        m = pd.concat([diag[keys + [\"FromStateClassId\", \"ToStateClassId\", \"Amount\"]], off],\n                      ignore_index=True)\n        m[\"OpeningTimestep\"], m[\"ClosingTimestep\"] = o, c\n        out.append(m)\n        col = m.groupby(keys + [\"ToStateClassId\"], as_index=False)[\"Amount\"].sum()\n        cl = st[st.Timestep == c].drop(columns=\"Timestep\").rename(\n            columns={\"StateClassId\": \"ToStateClassId\", \"Amount\": \"Closing\"})\n        chk = col.merge(cl, on=keys + [\"ToStateClassId\"], how=\"outer\").fillna(0.0)\n        chk[\"Residual\"] = (chk.Amount - chk.Closing).abs()\n        chk = chk.groupby(keys, as_index=False)[\"Residual\"].sum()\n        chk = chk.assign(Account=\"Change matrix\", OpeningTimestep=o, ClosingTimestep=c,\n                         Item=\"Column totals minus closing extent (should be 0)\")\n        checks.append(chk)\n    return _finish(pd.concat(out, ignore_index=True)), pd.concat(checks, ignore_index=True)\n\n\n# ---------------------------------------------------------------- carbon\n\nSTOCK_ENTRIES = [\n    \"Opening stock\",\n    \"Additions: sequestration\",\n    \"Additions: other inputs\",\n    \"Transfers in from other pools\",\n    \"Reductions: emissions (respiration and decomposition)\",\n    \"Reductions: emissions (conversion and disturbance)\",\n    \"Reductions: export to water\",\n    \"Reductions: transfer to long-term burial\",\n    \"Reductions: other losses\",\n    \"Transfers out to other pools\",\n    \"Change from ecosystem conversion\",\n    \"Net change in stock\",\n    \"Closing stock\",\n]\n\n\ndef stock_type_roles(groups: pd.DataFrame, membership: pd.DataFrame):\n    \"\"\"\n    groups: StockGroupId, Role. membership: StockTypeId, StockGroupId\n    (stsim_StockTypeGroupMembership). Returns ({type: role}, {pool type: pool group}).\n    An auto-generated group named '<type> [Type]' with no membership rows is\n    read as containing that one type.\n    \"\"\"\n    role_of, pool_of = {}, {}\n    for _, g in groups.iterrows():\n        gid, role = g[\"StockGroupId\"], list_value(g[\"Role\"], ROLES, \"carbon stock group role\")\n        types = list(membership.loc[membership[\"StockGroupId\"] == gid, \"StockTypeId\"])\n        if not types and isinstance(gid, str) and gid.endswith(\" [Type]\"):\n            types = [gid[: -len(\" [Type]\")]]\n        if not types:\n            raise AccountError(\n                f\"Carbon stock group '{gid}' has no stock types in Stock Type-Group \"\n                f\"Membership, so its flows cannot be classified.\")\n        for t in types:\n            if t in role_of and (role_of[t] != role or role == POOL):\n                raise AccountError(\n                    f\"Stock type '{t}' is in more than one listed carbon stock group. \"\n                    f\"List groups that do not overlap (e.g. AGB, BGB, Soil, not also Total).\")\n            role_of[t] = role\n            if role == POOL:\n                pool_of[t] = gid\n    return role_of, pool_of\n\n\ndef select_flow_rows(flow: pd.DataFrame, flow_groups: list, flow_membership=None):\n    \"\"\"\n    OutputFlow has one row per flow group, so a flow type in two groups is\n    reported twice. Use only the listed groups (they must not share flow\n    types) or, if none are listed, ST-Sim's auto groups '<type> [Type]'.\n    \"\"\"\n    if flow_groups:\n        if flow_membership is not None and not flow_membership.empty:\n            fm = flow_membership[flow_membership[\"FlowGroupId\"].isin(flow_groups)]\n            dup = fm[fm.duplicated(\"FlowTypeId\", keep=False)]\n            if not dup.empty:\n                raise AccountError(\n                    \"These flow types are in more than one listed carbon flow group and \"\n                    f\"would be counted twice: {sorted(map(str, dup.FlowTypeId.unique()))}\")\n        rows = flow[flow[\"FlowGroupId\"].isin(flow_groups)]\n    else:\n        auto = flow[\"FlowGroupId\"].astype(str).str.endswith(\" [Type]\")\n        if not auto.any():\n            raise AccountError(\n                \"No Carbon Flow Groups listed and no auto flow groups ('<type> [Type]') in \"\n                \"stsim_OutputFlow. List flow groups that together hold every carbon flow once.\")\n        rows = flow[auto]\n    return rows\n\n\ndef classify_flows(flow, role_of, by_stratum):\n    _need(flow, [\"Iteration\", \"Timestep\", \"FromStratumId\", \"FromStateClassId\", \"FromStockTypeId\",\n                 \"ToStockTypeId\", \"TransitionTypeId\", \"Amount\"], \"stsim_OutputFlow\")\n    if flow[\"FromStockTypeId\"].isna().all() or flow[\"ToStockTypeId\"].isna().all():\n        raise AccountError(\n            \"stsim_OutputFlow has no From/To stock types. In Stock-Flow Output options, \"\n            \"untick 'omit From stock type' and 'omit To stock type'.\")\n    df = _stratum(flow, \"FromStratumId\", by_stratum).rename(columns={\"FromStateClassId\": \"StateClassId\"})\n    df[\"FromRole\"] = df[\"FromStockTypeId\"].map(role_of).fillna(OTHER)\n    df[\"ToRole\"] = df[\"ToStockTypeId\"].map(role_of).fillna(OTHER)\n    fp, tp = df.FromRole == POOL, df.ToRole == POOL\n    by_transition = df[\"TransitionTypeId\"].notna()\n    cat = np.select(\n        [fp & tp,\n         ~fp & tp & (df.FromRole == ATMOSPHERE),\n         ~fp & tp,\n         fp & (df.ToRole == ATMOSPHERE) & by_transition,\n         fp & (df.ToRole == ATMOSPHERE),\n         fp & (df.ToRole == WATER),\n         fp & (df.ToRole == BURIAL),\n         fp],\n        [\"Internal transfer\",\n         \"Sequestration\",\n         \"Other inputs\",\n         \"Emissions (conversion and disturbance)\",\n         \"Emissions (respiration and decomposition)\",\n         \"Export to water\",\n         \"Transfer to long-term burial\",\n         \"Other losses\"],\n        default=\"\")\n    df[\"Category\"] = cat\n    return df[df.Category != \"\"]\n\n\ndef carbon_accounts(stock, flow, periods, groups, membership, flow_groups=None,\n                    flow_membership=None, by_stratum=False, log=None):\n    \"\"\"Returns (stock account, flow account, checks) or (None, None, None).\"\"\"\n    if groups is None or groups.empty:\n        _log(log, \"SEEA-Sim: no Carbon Stock Groups listed; carbon accounts skipped.\")\n        return None, None, None\n    role_of, pool_of = stock_type_roles(groups, membership)\n    pools = set(pool_of.values())\n    if not pools:\n        raise AccountError(\"No carbon stock group has Role = Ecosystem pool.\")\n    _need(stock, [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"StockGroupId\", \"Amount\"],\n          \"stsim_OutputStock\")\n    st = _stratum(stock[stock[\"StockGroupId\"].isin(pools)], \"StratumId\", by_stratum)\n    st = st.groupby([\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\", \"StockGroupId\"],\n                    as_index=False)[\"Amount\"].sum()\n    fl = classify_flows(select_flow_rows(flow, flow_groups, flow_membership), role_of, by_stratum)\n\n    keys = [\"Iteration\", \"StratumId\", \"StateClassId\", \"StockGroupId\"]\n    stock_rows, flow_rows, checks = [], [], []\n    for o, c in periods:\n        _check_timesteps(st, [o, c], \"stsim_OutputStock\")\n        f = fl[(fl.Timestep > o) & (fl.Timestep <= c)]\n\n        fa = (f.groupby([\"Iteration\", \"StratumId\", \"StateClassId\", \"TransitionTypeId\",\n                         \"FromStockTypeId\", \"ToStockTypeId\", \"Category\"],\n                        as_index=False, dropna=False)[\"Amount\"].sum())\n        fa[\"OpeningTimestep\"], fa[\"ClosingTimestep\"] = o, c\n        flow_rows.append(fa)\n\n        inflow = f[f.ToRole == POOL].assign(StockGroupId=lambda d: d.ToStockTypeId.map(pool_of))\n        inflow[\"Entry\"] = inflow.Category.map({\n            \"Sequestration\": \"Additions: sequestration\",\n            \"Other inputs\": \"Additions: other inputs\",\n            \"Internal transfer\": \"Transfers in from other pools\"})\n        outflow = f[f.FromRole == POOL].assign(StockGroupId=lambda d: d.FromStockTypeId.map(pool_of))\n        outflow[\"Entry\"] = outflow.Category.map(\n            lambda x: \"Transfers out to other pools\" if x == \"Internal transfer\"\n            else f\"Reductions: {x[0].lower()}{x[1:]}\")\n        outflow = outflow.assign(Amount=-outflow.Amount)\n        moves = (pd.concat([inflow, outflow])\n                 .groupby(keys + [\"Entry\"])[\"Amount\"].sum().unstack(\"Entry\"))\n\n        op = st[st.Timestep == o].set_index(keys)[\"Amount\"].rename(\"Opening stock\")\n        cl = st[st.Timestep == c].set_index(keys)[\"Amount\"].rename(\"Closing stock\")\n        w = pd.concat([op, cl, moves], axis=1).fillna(0.0)\n        for e in STOCK_ENTRIES:\n            if e not in w.columns:\n                w[e] = 0.0\n        change_cols = [e for e in STOCK_ENTRIES[1:10]]\n        w[\"Net change in stock\"] = w[\"Closing stock\"] - w[\"Opening stock\"]\n        w[\"Change from ecosystem conversion\"] = w[\"Net change in stock\"] - w[change_cols].sum(axis=1)\n        w = w[STOCK_ENTRIES]\n        w.columns.name = \"Entry\"\n        long = w.stack().rename(\"Amount\").reset_index()\n        long[\"OpeningTimestep\"], long[\"ClosingTimestep\"] = o, c\n        stock_rows.append(long)\n\n        # Over all classes the conversion term must cancel.\n        tot = (w[\"Change from ecosystem conversion\"].groupby([\"Iteration\", \"StratumId\"]).sum()\n               .rename(\"Residual\").reset_index())\n        checks.append(tot.assign(Account=\"Carbon stock\", OpeningTimestep=o, ClosingTimestep=c,\n                                 Item=\"Change from ecosystem conversion summed over classes \"\n                                      \"and pools (should be 0)\"))\n    stock_acc = pd.concat(stock_rows, ignore_index=True)\n    stock_acc[\"Entry\"] = pd.Categorical(stock_acc[\"Entry\"], STOCK_ENTRIES, ordered=True)\n    stock_acc = stock_acc.sort_values(keys[:2] + PERIOD + keys[2:] + [\"Entry\"])\n    stock_acc[\"Entry\"] = stock_acc[\"Entry\"].astype(str)\n    return (_finish(stock_acc), _finish(pd.concat(flow_rows, ignore_index=True)),\n            pd.concat(checks, ignore_index=True))\n\n\n# ---------------------------------------------------------------- services\n\ndef service_supply(state, stock, flow, rules, by_stratum=False, log=None):\n    \"\"\"\n    Physical supply per (Iteration, Timestep, StratumId, StateClassId, ServiceId).\n    Basis Area:        Coefficient x class area           (units per area unit per year)\n    Basis Stock group: Coefficient x stock of the group   (e.g. retention)\n    Basis Flow group:  Coefficient x flows of the group, attributed to FromStateClassId\n    Rules can be limited to one stratum and/or one state class.\n    The run's first timestep is the initial condition and has no flows, so\n    supply starts at the second timestep for every basis.\n    \"\"\"\n    if rules is None or rules.empty:\n        return None\n    first = state[\"Timestep\"].min()\n    parts = []\n    for _, r in rules.iterrows():\n        basis = list_value(r[\"Basis\"], BASES, \"service supply basis\")\n        if basis == AREA:\n            src = state\n            scol, ccol = \"StratumId\", \"StateClassId\"\n        elif basis == STOCK:\n            if pd.isna(r.get(\"StockGroupId\")):\n                raise AccountError(f\"Service '{r.ServiceId}': basis Stock group needs a Stock Group.\")\n            if stock is None:\n                raise AccountError(\"Stock-based service but stsim_OutputStock is empty.\")\n            src = stock[stock.StockGroupId == r.StockGroupId]\n            scol, ccol = \"StratumId\", \"StateClassId\"\n        else:\n            if pd.isna(r.get(\"FlowGroupId\")):\n                raise AccountError(f\"Service '{r.ServiceId}': basis Flow group needs a Flow Group.\")\n            if flow is None:\n                raise AccountError(\"Flow-based service but stsim_OutputFlow is empty.\")\n            src = flow[flow.FlowGroupId == r.FlowGroupId]\n            scol, ccol = \"FromStratumId\", \"FromStateClassId\"\n        if pd.notna(r.get(\"StratumId\")):\n            src = src[src[scol] == r.StratumId]\n        if pd.notna(r.get(\"StateClassId\")):\n            src = src[src[ccol] == r.StateClassId]\n        src = src[src.Timestep > first]\n        if src.empty:\n            continue\n        d = _stratum(src, scol, by_stratum).rename(columns={ccol: \"StateClassId\"})\n        d = d.groupby([\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\"], as_index=False)[\"Amount\"].sum()\n        d[\"Amount\"] *= float(r[\"Coefficient\"])\n        d[\"ServiceId\"] = r[\"ServiceId\"]\n        parts.append(d)\n    if not parts:\n        _log(log, \"SEEA-Sim: service supply rules matched no output rows.\")\n        return None\n    out = (pd.concat(parts).groupby([\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"ServiceId\"],\n                                    as_index=False)[\"Amount\"].sum())\n    return out\n\n\ndef price_table(prices: pd.DataFrame, services, timesteps, log=None) -> pd.DataFrame:\n    \"\"\"\n    ServiceId x Timestep -> Price. A price row applies from its Timestep\n    onward until the next one; a row with no Timestep applies before the\n    first dated row (or always, if none is dated).\n    \"\"\"\n    rows = []\n    for s in services:\n        p = prices[prices.ServiceId == s] if prices is not None and not prices.empty else prices\n        if p is None or p.empty:\n            _log(log, f\"SEEA-Sim: no price for service '{s}'; left out of monetary tables.\")\n            continue\n        dated = p[p[\"Timestep\"].notna()].sort_values(\"Timestep\") if \"Timestep\" in p else p.iloc[0:0]\n        undated = p[p[\"Timestep\"].isna()] if \"Timestep\" in p else p\n        for t in timesteps:\n            hit = dated[dated.Timestep <= t]\n            if not hit.empty:\n                price = hit.iloc[-1].Price\n            elif not undated.empty:\n                price = undated.iloc[0].Price\n            else:\n                price = dated.iloc[0].Price  # before the first dated price: use it\n            rows.append({\"ServiceId\": s, \"Timestep\": t, \"Price\": float(price)})\n    return pd.DataFrame(rows, columns=[\"ServiceId\", \"Timestep\", \"Price\"])\n\n\ndef service_value(supply, prices_tbl):\n    if supply is None or prices_tbl is None or prices_tbl.empty:\n        return None\n    v = supply.merge(prices_tbl, on=[\"ServiceId\", \"Timestep\"], how=\"inner\")\n    v[\"Amount\"] = v[\"Amount\"] * v[\"Price\"]\n    return v.drop(columns=\"Price\")\n\n\n# ---------------------------------------------------------------- assets\n\nASSET_ENTRIES = [\n    \"Opening value\",\n    \"Ecosystem enhancement\",\n    \"Ecosystem degradation\",\n    \"Ecosystem conversions: additions\",\n    \"Ecosystem conversions: reductions\",\n    \"Catastrophic losses\",\n    \"Upward reappraisals\",\n    \"Downward reappraisals\",\n    \"Other changes in volume n.e.c.\",\n    \"Revaluations\",\n    \"Net change in value\",\n    \"Closing value\",\n]\n\n\ndef condition_lookup(condition: pd.DataFrame | None):\n    \"\"\"\n    condition: StateClassId, optional OpeningTimestep/ClosingTimestep, Change.\n    Returns f(class, o, c) -> change or None. A row for the exact period wins\n    over a row with no period.\n    \"\"\"\n    if condition is None or condition.empty:\n        return lambda cls, o, c: None\n    d = condition.copy()\n    for col in (\"OpeningTimestep\", \"ClosingTimestep\"):\n        if col not in d.columns:\n            d[col] = np.nan\n    exact, anyp = {}, {}\n    for r in d.itertuples():\n        if pd.notna(r.OpeningTimestep) and pd.notna(r.ClosingTimestep):\n            exact[(r.StateClassId, int(r.OpeningTimestep), int(r.ClosingTimestep))] = float(r.Change)\n        else:\n            anyp[r.StateClassId] = float(r.Change)\n    return lambda cls, o, c: exact.get((cls, o, c), anyp.get(cls))\n\n\ndef classify_volume(volume: float, cond):\n    \"\"\"\n    SEEA EA Annex 10.1 (as tabulated in the SEEALand example):\n      volume > 0 and condition > 0          -> enhancement\n      volume > 0 and condition <= 0         -> upward reappraisal\n      volume < 0 and condition < 0          -> degradation\n      volume < 0 and condition >= 0         -> downward reappraisal\n    With no condition data, a gain is enhancement and a loss degradation.\n    \"\"\"\n    if volume > 0:\n        return \"Ecosystem enhancement\" if cond is None or cond > 0 else \"Upward reappraisals\"\n    if volume < 0:\n        return \"Ecosystem degradation\" if cond is None or cond < 0 else \"Downward reappraisals\"\n    return None\n\n\ndef asset_account(comp, state, supply, prices_tbl, valuation, by_stratum=False, log=None,\n                  condition=None):\n    \"\"\"\n    Monetary ecosystem asset account (SEEA EA ch. 10 and Annex 10.1).\n\n    Per class c and service s, the value is NPV = P x u x a, where\n      a = class area,\n      u = annual supply per area unit (expected flow intensity),\n      P = price x annuity(r, L), i.e. today's flow is assumed to continue for\n          the asset life.\n    Opening: a = opening area, u = supply per area in the period's first year\n    (o+1), price of year o. Closing: closing area, supply per area in year c,\n    price of year c. If a class has no area in one of those years, the other\n    year's u is used.\n\n    The change P1u1a1 - P0u0a0 is split exactly into area, volume and price\n    effects with the symmetric weights of SEEA EA Annex 10.1:\n      area   = (P0u0/3 + P0u1/6 + P1u0/6 + P1u1/3) x (a1 - a0)\n      volume = (P0a0/3 + P0a1/6 + P1a0/6 + P1a1/3) x (u1 - u0)\n      price  = (u0a0/3 + u0a1/6 + u1a0/6 + u1a1/3) x (P1 - P0)\n    The area effect is split by the extent account into conversions\n    (additions, reductions), catastrophic losses and unexplained change; the\n    volume effect (summed over services) into enhancement, degradation or\n    reappraisals using the condition change (classify_volume); the price\n    effect is revaluation.\n    \"\"\"\n    if supply is None or prices_tbl is None or prices_tbl.empty:\n        _log(log, \"SEEA-Sim: no priced services; asset account skipped.\")\n        return None\n    if valuation is None or valuation.empty:\n        _log(log, \"SEEA-Sim: Asset Valuation is empty; asset account skipped.\")\n        return None\n    default = valuation[valuation[\"StateClassId\"].isna()]\n    per_class = valuation[valuation[\"StateClassId\"].notna()].set_index(\"StateClassId\")\n\n    def af(cls):\n        if cls in per_class.index:\n            r = per_class.loc[cls]\n        elif not default.empty:\n            r = default.iloc[0]\n        else:\n            return np.nan\n        return annuity_factor(float(r.DiscountRate), int(r.AssetLifeYears))\n\n    cond_of = condition_lookup(condition)\n    if condition is None or condition.empty:\n        _log(log, \"SEEA-Sim: no condition data; volume gains are booked as enhancement and \"\n                  \"losses as degradation (no reappraisals).\")\n\n    st = _prep_state(state, by_stratum)\n    keys = [\"Iteration\", \"StratumId\", \"StateClassId\"]\n    q = supply.merge(st, on=[\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\"],\n                     how=\"left\", suffixes=(\"\", \"_area\"))\n    q[\"PerArea\"] = np.where(q[\"Amount_area\"] > 0, q[\"Amount\"] / q[\"Amount_area\"], np.nan)\n    prices = prices_tbl.set_index([\"ServiceId\", \"Timestep\"])[\"Price\"]\n    q = q[q.ServiceId.isin(set(prices_tbl.ServiceId))]\n\n    out = []\n    for (o, c), comp_p in comp.groupby(PERIOD):\n        o, c = int(o), int(c)\n        t0, t1 = o + 1, c\n        u0 = q[q.Timestep == t0].set_index(keys + [\"ServiceId\"])[\"PerArea\"]\n        u1 = q[q.Timestep == t1].set_index(keys + [\"ServiceId\"])[\"PerArea\"]\n        s = pd.concat([u0.rename(\"u0\"), u1.rename(\"u1\")], axis=1)\n        s[\"u0\"] = s[\"u0\"].fillna(s[\"u1\"])\n        s[\"u1\"] = s[\"u1\"].fillna(s[\"u0\"])\n        s = s.fillna(0.0).reset_index()\n        s[\"AF\"] = s.StateClassId.map(af)\n        if s.AF.isna().any():\n            miss = sorted(map(str, s.loc[s.AF.isna(), \"StateClassId\"].unique()))\n            _log(log, f\"SEEA-Sim: no discount rate/asset life for {miss}; valued at 0. \"\n                      \"Add a row with no State Class as the default.\")\n            s[\"AF\"] = s.AF.fillna(0.0)\n        s[\"P0\"] = [prices.get((sv, o), np.nan) for sv in s.ServiceId]\n        s[\"P1\"] = [prices.get((sv, c), np.nan) for sv in s.ServiceId]\n        s[[\"P0\", \"P1\"]] = s[[\"P0\", \"P1\"]].fillna(0.0).mul(s.AF, axis=0)\n        s = s.merge(comp_p[keys + [\"Opening\", \"Closing\"]], on=keys, how=\"left\").fillna(\n            {\"Opening\": 0.0, \"Closing\": 0.0})\n        P0, P1, a0, a1, U0, U1 = s.P0, s.P1, s.Opening, s.Closing, s.u0, s.u1\n        s[\"V0\"] = P0 * U0 * a0\n        s[\"V1\"] = P1 * U1 * a1\n        s[\"w_area\"] = P0 * U0 / 3 + P0 * U1 / 6 + P1 * U0 / 6 + P1 * U1 / 3\n        s[\"vol\"] = (P0 * a0 / 3 + P0 * a1 / 6 + P1 * a0 / 6 + P1 * a1 / 3) * (U1 - U0)\n        s[\"price\"] = (U0 * a0 / 3 + U0 * a1 / 6 + U1 * a0 / 6 + U1 * a1 / 3) * (P1 - P0)\n        cls = s.groupby(keys)[[\"V0\", \"V1\", \"w_area\", \"vol\", \"price\"]].sum().reset_index()\n        d = comp_p.merge(cls, on=keys, how=\"left\").fillna(\n            {\"V0\": 0.0, \"V1\": 0.0, \"w_area\": 0.0, \"vol\": 0.0, \"price\": 0.0})\n        w = d.w_area\n        vals = {e: pd.Series(0.0, index=d.index) for e in ASSET_ENTRIES}\n        vals[\"Opening value\"] = d.V0\n        vals[\"Closing value\"] = d.V1\n        vals[\"Net change in value\"] = d.V1 - d.V0\n        vals[\"Ecosystem conversions: additions\"] = (d.Add_Managed + d.Add_Unmanaged + d.Add_Catastrophic) * w\n        vals[\"Ecosystem conversions: reductions\"] = -(d.Red_Managed + d.Red_Unmanaged) * w\n        vals[\"Catastrophic losses\"] = -d.Red_Catastrophic * w\n        vals[\"Other changes in volume n.e.c.\"] = d.Residual * w\n        vals[\"Revaluations\"] = d.price\n        for i, r in d.iterrows():\n            entry = classify_volume(r.vol, cond_of(r.StateClassId, o, c))\n            if entry:\n                vals[entry].loc[i] = r.vol\n        base = d[[\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\"]]\n        out += [base.assign(Entry=e, Amount=vals[e].values) for e in ASSET_ENTRIES]\n    return _finish(pd.concat(out, ignore_index=True))\n\n\ndef account_checks(table: pd.DataFrame, account: str, opening: str, closing: str,\n                   net: str) -> pd.DataFrame:\n    \"\"\"Opening + sum(change entries) - closing, and net - (closing - opening).\"\"\"\n    t = table.copy()\n    t[\"StratumId\"] = t[\"StratumId\"].astype(object).where(t[\"StratumId\"].notna(), ALL)\n    idx = [c for c in [\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\", \"StockGroupId\"]\n           if c in t.columns]\n    w = t.pivot_table(index=idx, columns=\"Entry\", values=\"Amount\", aggfunc=\"sum\").fillna(0.0)\n    changes = [c for c in w.columns if c not in (opening, closing, net)]\n    res = (w[opening] + w[changes].sum(axis=1) - w[closing]).abs()\n    res = res.groupby([\"Iteration\", \"StratumId\", *PERIOD]).sum().rename(\"Residual\").reset_index()\n    return res.assign(Account=account, Item=\"Opening + changes - closing (should be 0)\")\n\n\ndef finish_checks(parts) -> pd.DataFrame:\n    parts = [p for p in parts if p is not None and not p.empty]\n    if not parts:\n        return pd.DataFrame(columns=[\"Iteration\", \"Account\", *PERIOD, \"StratumId\", \"Item\", \"Residual\"])\n    out = pd.concat(parts, ignore_index=True)\n    return _finish(out[[\"Iteration\", \"Account\", *PERIOD, \"StratumId\", \"Item\", \"Residual\"]])\n",
 "stsimseea.yml": "name: stsimseea\nchannels:\n  - conda-forge\n  - nodefaults\ndependencies:\n  - python=3.11\n  - pandas>=2.2\n  - numpy\n  - pysyncrosim>=2.1\n",
 "transformer.py": "\"\"\"\nSEEA-Sim transformer: reads ST-Sim output from the running result scenario,\nbuilds the SEEA EA accounts (seea_accounts.py) and saves them.\n\nRuns as the second pipeline stage, after stsim_Runtime.\n\"\"\"\nimport os\nimport sys\n\nimport pandas as pd\nimport pysyncrosim as ps\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport seea_accounts as sa  # noqa: E402\n\nPKG = \"stsimseea\"\n\n\ndef read(scn, name, empty_ok=True):\n    try:\n        df = scn.datasheets(name=name)\n    except Exception as e:  # sheet missing from this library\n        if empty_ok:\n            return pd.DataFrame()\n        raise sa.AccountError(f\"Could not read {name}: {e}\")\n    if (df is None or df.empty) and not empty_ok:\n        raise sa.AccountError(f\"{name} is empty. Check that ST-Sim wrote this output.\")\n    return df if df is not None else pd.DataFrame()\n\n\ndef save(scn, name, df, columns):\n    if df is None or df.empty:\n        return\n    scn.save_datasheet(name=f\"{PKG}_{name}\", data=df[[c for c in columns if c in df.columns]])\n    print(f\"SEEA-Sim: saved {name} ({len(df)} rows)\")\n\n\ndef check_output_options(scn):\n    \"\"\"Fail early with a clear message if ST-Sim did not write what we need.\"\"\"\n    oo = read(scn, \"stsim_OutputOptions\")\n    if oo.empty:\n        return\n    r = oo.iloc[0]\n    if not bool(r.get(\"SummaryOutputTRSC\", False)):\n        raise sa.AccountError(\n            \"Turn on ST-Sim Output Options > Tabular > Transitions by state class \"\n            \"(SummaryOutputTRSC), every 1 timestep. The extent account is built from it.\")\n    if pd.notna(r.get(\"SummaryOutputTRSCTimesteps\")) and int(r[\"SummaryOutputTRSCTimesteps\"]) != 1:\n        raise sa.AccountError(\"Set the Transitions by state class output frequency to 1 timestep.\")\n\n\ndef main():\n    scn = ps.Scenario()\n\n    opts = read(scn, f\"{PKG}_Options\")\n    by_stratum = bool(opts.iloc[0].get(\"ReportByStratum\", False)) if not opts.empty else False\n\n    check_output_options(scn)\n    state = read(scn, \"stsim_OutputStratumState\", empty_ok=False)\n    trans = read(scn, \"stsim_OutputStratumTransitionState\")\n    if trans.empty:\n        trans = pd.DataFrame(columns=[\"Iteration\", \"Timestep\", \"StratumId\", \"TransitionTypeId\",\n                                      \"StateClassId\", \"EndStateClassId\", \"Amount\"])\n\n    per = read(scn, f\"{PKG}_AccountingPeriod\")\n    periods = ([(int(r.OpeningTimestep), int(r.ClosingTimestep)) for r in per.itertuples()]\n               if not per.empty else sa.default_periods(state))\n    for o, c in periods:\n        if c <= o:\n            raise sa.AccountError(f\"Accounting period {o}-{c}: closing must be after opening.\")\n    print(f\"SEEA-Sim: periods {periods}; by stratum = {by_stratum}\")\n\n    ttc = read(scn, f\"{PKG}_TransitionTypeClass\")\n    tt_class = ({r.TransitionTypeId: sa.list_value(r.ChangeType, sa.CHANGE_TYPES, \"change type\")\n                 for r in ttc.itertuples()} if not ttc.empty else {})\n\n    # Extent account and change matrix\n    comp = sa.extent_components(state, trans, periods, tt_class, by_stratum)\n    extent = sa.extent_account(comp)\n    matrix, matrix_chk = sa.change_matrix(state, trans, periods, tt_class, by_stratum)\n    period_cols = [\"OpeningTimestep\", \"ClosingTimestep\", \"StratumId\"]\n    save(scn, \"OutputExtent\", extent, [\"Iteration\", *period_cols, \"StateClassId\", \"Entry\", \"Amount\"])\n    save(scn, \"OutputChangeMatrix\", matrix,\n         [\"Iteration\", *period_cols, \"FromStateClassId\", \"ToStateClassId\", \"Amount\"])\n    checks = [sa.account_checks(extent, \"Extent\", \"Opening extent\", \"Closing extent\",\n                                \"Net change in extent\"),\n              comp.assign(Residual=comp.Residual.abs())\n                  .groupby([\"Iteration\", \"StratumId\", \"OpeningTimestep\", \"ClosingTimestep\"],\n                           as_index=False)[\"Residual\"].sum()\n                  .assign(Account=\"Extent\",\n                          Item=\"Area change not explained by recorded transitions\"),\n              matrix_chk]\n\n    # Carbon\n    stock = read(scn, \"stsim_OutputStock\")\n    flow = read(scn, \"stsim_OutputFlow\")\n    groups = read(scn, f\"{PKG}_CarbonStockGroup\")\n    if not groups.empty:\n        if stock.empty or flow.empty:\n            raise sa.AccountError(\n                \"Carbon Stock Groups are set but stsim_OutputStock or stsim_OutputFlow is empty. \"\n                \"Turn on Stock-Flow summary output (stocks and flows, every timestep).\")\n        membership = read(scn, \"stsim_StockTypeGroupMembership\")\n        if membership.empty:\n            membership = pd.DataFrame(columns=[\"StockTypeId\", \"StockGroupId\"])\n        fgroups = list(read(scn, f\"{PKG}_CarbonFlowGroup\").get(\"FlowGroupId\", []))\n        fmember = read(scn, \"stsim_FlowTypeGroupMembership\")\n        cstock, cflow, cchk = sa.carbon_accounts(stock, flow, periods, groups, membership,\n                                                 fgroups, fmember, by_stratum)\n        save(scn, \"OutputCarbonStock\", cstock,\n             [\"Iteration\", *period_cols, \"StateClassId\", \"StockGroupId\", \"Entry\", \"Amount\"])\n        save(scn, \"OutputCarbonFlow\", cflow,\n             [\"Iteration\", *period_cols, \"StateClassId\", \"TransitionTypeId\", \"FromStockTypeId\",\n              \"ToStockTypeId\", \"Category\", \"Amount\"])\n        checks += [cchk, sa.account_checks(cstock, \"Carbon stock\", \"Opening stock\",\n                                           \"Closing stock\", \"Net change in stock\")]\n    else:\n        print(\"SEEA-Sim: no Carbon Stock Groups; carbon accounts skipped.\")\n\n    # Ecosystem services and assets\n    rules = read(scn, f\"{PKG}_ServiceSupply\")\n    supply = sa.service_supply(state, stock if not stock.empty else None,\n                               flow if not flow.empty else None, rules, by_stratum)\n    if supply is not None:\n        out_cols = [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"ServiceId\", \"Amount\"]\n        save(scn, \"OutputServiceSupply\", sa._finish(supply.copy()), out_cols)\n        prices = read(scn, f\"{PKG}_ServicePrice\")\n        if not prices.empty and \"Timestep\" not in prices.columns:\n            prices[\"Timestep\"] = pd.NA\n        price_ts = sorted(set(supply.Timestep.unique()) | {o for o, _ in periods})\n        ptbl = sa.price_table(prices, supply.ServiceId.unique(), price_ts)\n        value = sa.service_value(supply, ptbl)\n        if value is not None:\n            save(scn, \"OutputServiceValue\", sa._finish(value.copy()), out_cols)\n        cond = read(scn, f\"{PKG}_ConditionChange\")\n        assets = sa.asset_account(comp, state, supply, ptbl, read(scn, f\"{PKG}_AssetValuation\"),\n                                  by_stratum, condition=cond if not cond.empty else None)\n        if assets is not None:\n            save(scn, \"OutputAssetAccount\", assets,\n                 [\"Iteration\", *period_cols, \"StateClassId\", \"Entry\", \"Amount\"])\n            checks.append(sa.account_checks(assets, \"Asset\", \"Opening value\", \"Closing value\",\n                                            \"Net change in value\"))\n    else:\n        print(\"SEEA-Sim: no Service Supply rules; service and asset accounts skipped.\")\n\n    chk = sa.finish_checks(checks)\n    save(scn, \"OutputCheck\", chk,\n         [\"Iteration\", \"Account\", *period_cols, \"Item\", \"Residual\"])\n    worst = chk.groupby([\"Account\", \"Item\"])[\"Residual\"].max() if not chk.empty else []\n    for (acc, item), v in getattr(worst, \"items\", lambda: [])():\n        print(f\"SEEA-Sim check: {acc}: {item}: max {v:.6g}\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except sa.AccountError as e:\n        print(f\"SEEA-Sim input error: {e}\")\n        raise\n"
}

In [ ]:
# 6b. Bundle it into stsimseea-1-0-3.ssimpkg (files at the zip root + .metadata) and install it
import re, zipfile
xml = PKG_FILES['package.xml']
ver = re.search(r'<package\s[^>]*version="([^"]+)"', xml).group(1)
ssim_min = re.search(r'minSyncroSimVersion="([^"]+)"', xml).group(1)
PKG_PATH = f"/content/stsimseea-{ver.replace('.', '-')}.ssimpkg"
with zipfile.ZipFile(PKG_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('.metadata', ('\ufeff<?xml version="1.0" encoding="utf-8"?>\r\n'
                             f'<package builtForSyncroSimVersion="{ssim_min}" />').encode('utf-8'))
    for name, text in PKG_FILES.items():
        z.writestr(name, text)
print(PKG_PATH, zipfile.ZipFile(PKG_PATH).namelist())
s.install_packages(PKG_PATH)
print(s.packages())

In [ ]:
# 7. Mark 0 as no-data in the start and stratum maps (local copy), so SyncroSim gets a cell area
import rasterio
R = os.path.join(WORK, 'rasters')
for f in ('2015.tif', 'stratum_2015.tif'):
    p = os.path.join(R, f)
    if os.path.exists(p):
        with rasterio.open(p, 'r+') as r:
            if r.nodata is None:
                r.nodata = 0
        print(f, 'nodata =', rasterio.open(p).nodata)

In [ ]:
# 8. Build the ST-Sim library (full set-up, no run)
%cd {WORK}
!python build_and_test.py --full --no-run --ssim /root/syncrosim
log = open(os.path.join(WORK, 'build_log.txt'), encoding='utf-8').read().splitlines()
bad = [l for l in log if l.startswith(('[FAIL', '[WARN'))]
print('\n'.join(bad) if bad else 'build_log: no FAIL or WARN lines')

In [ ]:
# 9. Check the cell area; re-import the initial conditions if SyncroSim stored 0
LIB = os.path.join(WORK, 'JEM_cycle1.ssim')
SCN = 'Cycle1 2015 BaU'
lib = ps.library(name=LIB, session=s)
scn = lib.projects(name='Definitions').scenarios(name=SCN)
p = scn.datasheets('stsim_InitialConditionsSpatialProperties')
print(p.T.to_string())
if 'CellArea' in p.columns and (p['CellArea'].isna().all() or float(p['CellArea'].iloc[0] or 0) == 0):
    ics = __import__('pandas').DataFrame({'StratumFileName': [os.path.join(R, 'stratum_2015.tif')],
                        'StateClassFileName': [os.path.join(R, '2015.tif')],
                        'AgeFileName': [os.path.join(R, 'age_2015.tif')]})
    scn.save_datasheet(name='stsim_InitialConditionsSpatial', data=ics)
    print('re-imported:', scn.datasheets('stsim_InitialConditionsSpatialProperties')[['CellArea']].T.to_string())

In [ ]:
# 10a. SEEA-Sim tables for JEM (same as examples/jem_demak in the repo)
SEEA_FILES = {
 "README.md": "# Example: JEM Demak mangrove coast (validation 2015 to 2022)\n\nSEEA-Sim inputs for the JEM ST-Sim model of the Demak coast (7 classes, 20 transition types, carbon stocks AGB, BGB and Soil). Names match that library; for another model, change the names and values.\n\n## Files\n\n| File | Level | What it sets |\n|---|---|---|\n| `project/stsimseea_Service.csv` | Project | 8 ecosystem services with type and physical unit |\n| `project/stsim_StockGroup_additions.csv` | Project (ST-Sim) | One stock group per carbon stock type (`SEEA_AGB` ... `SEEA_Buried`). The model's own groups overlap (Total and Biomass both hold AGB), and the carbon accounts need groups that don't |\n| `project/stsim_FlowGroup_additions.csv` | Project (ST-Sim) | `SEEA_all_carbon_flows` (every flow type once) and `SEEA_NPP` (the GPP flow type, for Carbon Sequestration) |\n| `scenario/stsim_StockTypeGroupMembership_additions.csv` | Scenario (ST-Sim) | Puts each stock type in its SEEA group |\n| `scenario/stsim_FlowTypeGroupMembership_additions.csv` | Scenario (ST-Sim) | Puts the flow types in the two SEEA flow groups |\n| `scenario/stsim_OutputOptions_required.csv` | Scenario (ST-Sim) | Turns on transitions-by-state-class output every timestep |\n| `scenario/stsimseea_Options.csv` | Scenario | Whole landscape (no stratum split), currency label |\n| `scenario/stsimseea_AccountingPeriod.csv` | Scenario | 2015 to 2020 and 2020 to 2022 |\n| `scenario/stsimseea_TransitionTypeClass.csv` | Scenario | 10 human conversions Managed, 10 natural processes Unmanaged |\n| `scenario/stsimseea_CarbonStockGroup.csv` | Scenario | AGB, BGB, Soil = ecosystem pools; Atmosphere, Water, Buried = external |\n| `scenario/stsimseea_CarbonFlowGroup.csv` | Scenario | Use `SEEA_all_carbon_flows` |\n| `scenario/stsimseea_ServiceSupply.csv` | Scenario | How each service's quantity is computed |\n| `scenario/stsimseea_ServicePrice.csv` | Scenario | IDR per physical unit |\n| `scenario/stsimseea_AssetValuation.csv` | Scenario | 3% discount rate; asset life 100 years (natural), 50 (settlement, cropland), 30 (ponds); same as `06_AssetValuationParams.csv` |\n| `add_seeasim.py` | | Loads all of the above into a library |\n\n## Load into a library\n\nAfter the ST-Sim library is built (for JEM: `python build_and_test.py --full --no-run`), and with `stsimseea` installed in SyncroSim:\n\n```\npython add_seeasim.py --library JEM_cycle1.ssim --scenario \"Cycle1 2015 BaU\" --ssim \"C:\\Program Files\\SyncroSim Studio\"\n```\n\nIt adds the package to the library, adds the SEEA groups and the service list, saves the scenario tables, turns on the output SEEA-Sim needs, and sets the pipeline (1 ST-Sim, 2 SEEA-Sim Accounts). Project groups and memberships are only added if missing, so running it twice is safe. Check `add_seeasim_log.txt` for FAIL lines.\n\n## Where the values come from, and what to check\n\nPhysical quantities and per-hectare values are the earlier SEEA-Sim drafts' values (`3_seeasim_addon` in the JEM inputs). Prices are those values divided by the quantity per hectare:\n\n| Service | Basis | Quantity | Price (IDR) | Same as |\n|---|---|---|---|---|\n| Capture Fisheries | Water body area | 150 kg/ha/yr | 20,000 /kg | 3.0 M/ha |\n| Crop Provisioning | Cropland area | 5,000 kg/ha/yr | 6,000 /kg | 30 M/ha |\n| Aquaculture Fishery | Pond area | 800 kg/ha/yr | 56,250 /kg | 45 M/ha |\n| Fishery Nursery | Mangrove area | 1 ha | 8 M /ha | 8 M/ha |\n| Coastal Protection | Mangrove area x 1, tidal flat x 1/3 | ha mangrove-equivalent | 15 M /ha | 15 M/ha mangrove, 5 M/ha tidal flat |\n| Carbon Sequestration | NPP flows (flow group `SEEA_NPP`) | MgC | 238,333 /MgC | 238,333 /MgC |\n| Ecotourism | Mangrove area | 1 ha | 12.5 M /ha | 12.5 M/ha |\n| Biodiversity Habitat | Other vegetation area | 1 ha | 10 M /ha | 10 M/ha |\n\nService names, units, the carbon price and the asset-valuation table (`DiscountRate`, `AssetLifeYears`, one row per class) follow `3_seeasim_addon` in the JEM inputs. `PriceGrowthRate` is not used: give dated rows in Service Prices instead.\n\nChanges from the earlier draft:\n- **Carbon sequestration on Other vegetation (350 MgC/ha)** is replaced by the model's own NPP flows for every class, so the unit question goes away.\n- **Carbon storage** (pricing the AGB, BGB and soil stock every year) is left out of the services: it would value the whole stock again each year. The stock is reported in the carbon stock account instead.\n- **Carbon Sequestration** here is gross uptake (NPP). Net sequestration would subtract soil respiration; decide which one the paper reports.\n\nBefore using these numbers in results:\n- The prices are placeholders from the drafts. Replace them with sourced Demak values and a stated price year.\n- Inundation is classed Unmanaged. If you treat abrasion as a disaster, change it to Catastrophic and it moves to \"Catastrophic losses\" in the asset account.\n- The validation scenario runs 2015 to 2022. For the projections, use 2022 to 2026 and 2026 to 2031 as periods.\n",
 "add_seeasim.py": "\"\"\"\nAdd the SEEA-Sim example inputs to an existing ST-Sim library.\n\n    python add_seeasim.py --library JEM_cycle1.ssim --scenario \"Cycle1 2015 BaU\"\n    python add_seeasim.py ... --ssim \"C:\\\\Program Files\\\\SyncroSim Studio\"\n\nNeeds the stsimseea package installed in SyncroSim (File > Local Packages).\nWhat it does, in order:\n  1. adds the stsimseea package to the library\n  2. project: adds the SEEA stock and flow groups (only names not there yet)\n     and the Ecosystem Service list\n  3. scenario: adds the stock/flow group memberships (keeps existing rows),\n     turns on the ST-Sim output SEEA-Sim needs, saves every stsimseea_* table\n  4. sets the scenario pipeline: 1 ST-Sim, 2 SEEA-Sim Accounts\nWrites add_seeasim_log.txt next to this script.\n\"\"\"\nimport argparse\nimport glob\nimport os\n\nimport pandas as pd\nimport pysyncrosim as ps\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nP_DIR = os.path.join(HERE, \"project\")\nS_DIR = os.path.join(HERE, \"scenario\")\nLOG = []\n\n# Codes for SyncroSim List columns, tried if saving the labels fails.\nLIST_CODES = {\n    \"ServiceType\": {\"Provisioning\": 1, \"Regulating\": 2, \"Cultural\": 3},\n    \"ChangeType\": {\"Managed\": 1, \"Unmanaged\": 2, \"Catastrophic\": 3},\n    \"Role\": {\"Ecosystem pool\": 1, \"Atmosphere\": 2, \"Water\": 3, \"Long-term burial\": 4,\n             \"Other external\": 5},\n    \"Basis\": {\"Area\": 1, \"Stock group\": 2, \"Flow group\": 3},\n}\n\n\ndef log(level, msg):\n    line = f\"[{level}] {msg}\"\n    print(line)\n    LOG.append(line)\n\n\ndef read_csv(path):\n    return pd.read_csv(path, dtype=str, keep_default_na=False).replace(\"\", None)\n\n\ndef norm(s):\n    return s.lower().replace(\"_\", \"\")\n\n\ndef match_columns(df, obj, name):\n    \"\"\"Rename CSV columns to the datasheet's own spelling (Id vs ID).\"\"\"\n    try:\n        real = list(obj.datasheets(name, empty=True).columns)\n    except Exception:\n        return df\n    lookup = {norm(c): c for c in real}\n    keep = {c: lookup[norm(c)] for c in df.columns if norm(c) in lookup}\n    dropped = [c for c in df.columns if c not in keep]\n    if dropped:\n        log(\"WARN\", f\"{name}: columns not in datasheet, dropped {dropped}; real columns {real}\")\n    return df[list(keep)].rename(columns=keep)\n\n\ndef save(obj, name, df, **kw):\n    df = match_columns(df, obj, name)\n    try:\n        obj.save_datasheet(name=name, data=df, **kw)\n        log(\"OK\", f\"{name}: {len(df)} rows\")\n        return\n    except Exception as e:\n        first = str(e).strip()[:300]\n    coded = df.copy()\n    changed = False\n    for col, codes in LIST_CODES.items():\n        if col in coded.columns:\n            coded[col] = coded[col].map(lambda v: codes.get(v, v) if v is not None else v)\n            changed = True\n    if changed:\n        try:\n            obj.save_datasheet(name=name, data=coded, **kw)\n            log(\"OK\", f\"{name}: {len(df)} rows (list values saved as codes)\")\n            return\n        except Exception as e:\n            first += \" | with codes: \" + str(e).strip()[:300]\n    log(\"FAIL\", f\"{name}: {first}\")\n\n\ndef add_new_names(proj, name, path):\n    \"\"\"Project definitions: append only names that are not there yet.\"\"\"\n    new = read_csv(path)\n    try:\n        cur = proj.datasheets(name)\n        have = set(cur[\"Name\"].astype(str)) if \"Name\" in cur.columns else set()\n    except Exception:\n        have = set()\n    new = new[~new[\"Name\"].isin(have)]\n    if new.empty:\n        log(\"OK\", f\"{name}: nothing new\")\n        return\n    save(proj, name, new, append=True)\n\n\ndef merge_membership(scn, name, path, key_cols):\n    new = match_columns(read_csv(path), scn, name)\n    try:\n        cur = scn.datasheets(name)\n    except Exception:\n        cur = pd.DataFrame(columns=new.columns)\n    both = pd.concat([cur[[c for c in new.columns if c in cur.columns]], new], ignore_index=True)\n    keys = [c for c in both.columns if norm(c) in {norm(k) for k in key_cols}]\n    both = both.drop_duplicates(subset=keys)\n    save(scn, name, both)\n\n\ndef require_output_options(scn, path):\n    name = \"stsim_OutputOptions\"\n    want = read_csv(path).iloc[0].to_dict()\n    try:\n        cur = scn.datasheets(name)\n    except Exception:\n        cur = pd.DataFrame()\n    row = cur.iloc[[0]].copy() if len(cur) else pd.DataFrame([{}])\n    for k, v in want.items():\n        row[k] = v\n    save(scn, name, row)\n\n\ndef set_pipeline(scn):\n    for first, second in ((\"stsim_Runtime\", \"stsimseea_Main\"), (\"ST-Sim\", \"SEEA-Sim Accounts\")):\n        try:\n            scn.save_datasheet(name=\"core_Pipeline\", data=pd.DataFrame(\n                {\"StageNameId\": [first, second], \"RunOrder\": [1, 2]}))\n            log(\"OK\", f\"pipeline: 1 {first}, 2 {second}\")\n            return\n        except Exception:\n            continue\n    log(\"WARN\", \"could not set the pipeline; set it in Studio: Scenario > Pipeline: \"\n                \"1 ST-Sim, 2 SEEA-Sim Accounts\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--library\", required=True)\n    ap.add_argument(\"--scenario\", required=True)\n    ap.add_argument(\"--project\", default=\"Definitions\")\n    ap.add_argument(\"--ssim\", default=None)\n    a = ap.parse_args()\n\n    session = ps.Session(location=a.ssim) if a.ssim else ps.Session()\n    lib = ps.library(name=a.library, session=session)\n    try:\n        lib.add_packages(\"stsimseea\")\n        log(\"OK\", \"package stsimseea added to the library\")\n    except Exception as e:\n        log(\"WARN\", f\"add_packages(stsimseea): {str(e)[:300]} (fine if it is already in the library)\")\n    proj = lib.projects(name=a.project)\n    scn = proj.scenarios(name=a.scenario)\n\n    add_new_names(proj, \"stsim_StockGroup\", os.path.join(P_DIR, \"stsim_StockGroup_additions.csv\"))\n    add_new_names(proj, \"stsim_FlowGroup\", os.path.join(P_DIR, \"stsim_FlowGroup_additions.csv\"))\n    add_new_names(proj, \"stsimseea_Service\", os.path.join(P_DIR, \"stsimseea_Service.csv\"))\n\n    merge_membership(scn, \"stsim_StockTypeGroupMembership\",\n                     os.path.join(S_DIR, \"stsim_StockTypeGroupMembership_additions.csv\"),\n                     [\"StockTypeId\", \"StockGroupId\"])\n    merge_membership(scn, \"stsim_FlowTypeGroupMembership\",\n                     os.path.join(S_DIR, \"stsim_FlowTypeGroupMembership_additions.csv\"),\n                     [\"FlowTypeId\", \"FlowGroupId\"])\n    require_output_options(scn, os.path.join(S_DIR, \"stsim_OutputOptions_required.csv\"))\n    for path in sorted(glob.glob(os.path.join(S_DIR, \"stsimseea_*.csv\"))):\n        save(scn, os.path.splitext(os.path.basename(path))[0], read_csv(path))\n    set_pipeline(scn)\n\n    with open(os.path.join(HERE, \"add_seeasim_log.txt\"), \"w\", encoding=\"utf-8\") as f:\n        f.write(\"\\n\".join(LOG))\n    n_fail = sum(line.startswith(\"[FAIL\") for line in LOG)\n    print(f\"\\nDone: {n_fail} FAIL. Details in add_seeasim_log.txt\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "project/stsim_FlowGroup_additions.csv": "Name,Description,Units\nSEEA_all_carbon_flows,Every carbon flow type once (used by the SEEA-Sim carbon accounts),MgC\nSEEA_NPP,NPP flows from the atmosphere into biomass (GPP flow type),MgC\n",
 "project/stsim_StockGroup_additions.csv": "Name,Description,Units\nSEEA_AGB,SEEA-Sim pool: aboveground biomass (only AGB),MgC\nSEEA_BGB,SEEA-Sim pool: belowground biomass (only BGB),MgC\nSEEA_Soil,SEEA-Sim pool: soil organic carbon (only Soil),MgC\nSEEA_Atmosphere,SEEA-Sim external: atmosphere,MgC\nSEEA_Water,SEEA-Sim external: adjacent water,MgC\nSEEA_Buried,SEEA-Sim external: permanent burial,MgC\n",
 "project/stsimseea_Service.csv": "Name,ServiceType,Units,Description\nCapture Fisheries,Provisioning,kg,Wild fish catch supported by coastal water (150 kg/ha/yr)\nCrop Provisioning,Provisioning,kg,Rice and other crops (5000 kg/ha/yr)\nAquaculture Fishery,Provisioning,kg,Pond fish and shrimp (800 kg/ha/yr)\nFishery Nursery,Provisioning,ha,Mangrove area serving as nursery habitat\nCoastal Protection,Regulating,ha mangrove-equivalent,Wave and erosion protection; tidal flat counts as 1/3 of mangrove\nCarbon Sequestration,Regulating,MgC,Carbon taken up by vegetation: NPP flows from the atmosphere (flow group SEEA_NPP)\nEcotourism,Cultural,ha,Mangrove area used for ecotourism\nBiodiversity Habitat,Cultural,ha,Semi-natural vegetation habitat\n",
 "scenario/stsim_FlowTypeGroupMembership_additions.csv": "FlowTypeId,FlowGroupId,Value\nGPP,SEEA_all_carbon_flows,\nEmission,SEEA_all_carbon_flows,\nHumification,SEEA_all_carbon_flows,\nExport,SEEA_all_carbon_flows,\nSoil Export,SEEA_all_carbon_flows,\nBurial,SEEA_all_carbon_flows,\nSedimentation,SEEA_all_carbon_flows,\nGPP,SEEA_NPP,\n",
 "scenario/stsim_OutputOptions_required.csv": "SummaryOutputSC,SummaryOutputSCTimesteps,SummaryOutputTR,SummaryOutputTRTimesteps,SummaryOutputTRSC,SummaryOutputTRSCTimesteps\nYes,1,Yes,1,Yes,1\n",
 "scenario/stsim_StockTypeGroupMembership_additions.csv": "StockTypeId,StockGroupId,Value\nAGB,SEEA_AGB,\nBGB,SEEA_BGB,\nSoil,SEEA_Soil,\nAtmosphere,SEEA_Atmosphere,\nWater,SEEA_Water,\nBuried,SEEA_Buried,\n",
 "scenario/stsimseea_AccountingPeriod.csv": "OpeningTimestep,ClosingTimestep\n2015,2020\n2020,2022\n",
 "scenario/stsimseea_AssetValuation.csv": "StateClassId,DiscountRate,AssetLifeYears\n,0.03,100\nWater_body:All,0.03,100\nTidal_flat:All,0.03,100\nMangrove:All,0.03,100\nOther_vegetation:All,0.03,100\nSettlement:All,0.03,50\nCropland:All,0.03,50\nAquaculture:All,0.03,30\n",
 "scenario/stsimseea_CarbonFlowGroup.csv": "FlowGroupId\nSEEA_all_carbon_flows\n",
 "scenario/stsimseea_CarbonStockGroup.csv": "StockGroupId,Role\nSEEA_AGB,Ecosystem pool\nSEEA_BGB,Ecosystem pool\nSEEA_Soil,Ecosystem pool\nSEEA_Atmosphere,Atmosphere\nSEEA_Water,Water\nSEEA_Buried,Long-term burial\n",
 "scenario/stsimseea_Options.csv": "ReportByStratum,Currency\nNo,IDR (2022 prices)\n",
 "scenario/stsimseea_ServicePrice.csv": "ServiceId,Timestep,Price\nCapture Fisheries,,20000\nCrop Provisioning,,6000\nAquaculture Fishery,,56250\nFishery Nursery,,8000000\nCoastal Protection,,15000000\nCarbon Sequestration,,238333\nEcotourism,,12500000\nBiodiversity Habitat,,10000000\n",
 "scenario/stsimseea_ServiceSupply.csv": "ServiceId,StratumId,StateClassId,Basis,StockGroupId,FlowGroupId,Coefficient\nCapture Fisheries,,Water_body:All,Area,,,150\nCrop Provisioning,,Cropland:All,Area,,,5000\nAquaculture Fishery,,Aquaculture:All,Area,,,800\nFishery Nursery,,Mangrove:All,Area,,,1\nCoastal Protection,,Mangrove:All,Area,,,1\nCoastal Protection,,Tidal_flat:All,Area,,,0.3333\nCarbon Sequestration,,,Flow group,,SEEA_NPP,1\nEcotourism,,Mangrove:All,Area,,,1\nBiodiversity Habitat,,Other_vegetation:All,Area,,,1\n",
 "scenario/stsimseea_TransitionTypeClass.csv": "TransitionTypeId,ChangeType\nUrbanization__Mangrove_to_Settlement,Managed\nAgriculture_expansion_Mangrove__Mangrove_to_Cropland,Managed\nAquaculture_expansion__Mangrove_to_Aquaculture,Managed\nUrbanization__Other_vegetation_to_Settlement,Managed\nAgriculture_expansion_OV__Other_vegetation_to_Cropland,Managed\nAquaculture_expansion__Other_vegetation_to_Aquaculture,Managed\nUrbanization__Cropland_to_Settlement,Managed\nAquaculture_expansion__Cropland_to_Aquaculture,Managed\nUrbanization__Aquaculture_to_Settlement,Managed\nAgriculture_expansion_Aqua__Aquaculture_to_Cropland,Managed\nSedimentation__Water_body_to_Tidal_flat,Unmanaged\nMangrove_recruitment__Water_body_to_Mangrove,Unmanaged\nTidal_flat_to_Waterbody__Tidal_flat_to_Water_body,Unmanaged\nMangrove_recruitment__Tidal_flat_to_Mangrove,Unmanaged\nInundation__Mangrove_to_Water_body,Unmanaged\nInundation__Mangrove_to_Tidal_flat,Unmanaged\nMangrove_recruitment__Cropland_to_Mangrove,Unmanaged\nColonization__Cropland_to_Other_vegetation,Unmanaged\nMangrove_recruitment__Aquaculture_to_Mangrove,Unmanaged\nColonization__Aquaculture_to_Other_vegetation,Unmanaged\n"
}

In [ ]:
# 10b. Write them to 4_seeasim/ and check every name against this folder's ST-Sim tables
import pandas as pd
OUT = os.path.join(WORK, '4_seeasim')
for rel, text in SEEA_FILES.items():
    path = os.path.join(OUT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8', newline='') as f:
        f.write(text)
P = os.path.join(WORK, '1_project_definitions')
names = lambda f: set(pd.read_csv(os.path.join(P, f), dtype=str)['Name'].str.strip())
classes, ttypes = names('05_stsim_StateClass.csv'), names('06_stsim_TransitionType.csv')
stypes, ftypes = names('13_stsim_StockType.csv'), names('16_stsim_FlowType.csv')
rd = lambda f: pd.read_csv(os.path.join(OUT, f), dtype=str, keep_default_na=False)
problems = []
def check(values, allowed, what):
    problems.extend(f'{what}: {v!r} is not in the model' for v in sorted(set(values) - {''} - allowed))
ttc = rd('scenario/stsimseea_TransitionTypeClass.csv')
check(ttc.TransitionTypeId, ttypes, 'Transition type')
problems += [f'Transition type {t!r} has no class (counts as Unmanaged)' for t in sorted(ttypes - set(ttc.TransitionTypeId))]
check(rd('scenario/stsimseea_ServiceSupply.csv').StateClassId, classes, 'Supply rule class')
check(rd('scenario/stsimseea_AssetValuation.csv').StateClassId, classes, 'Asset valuation class')
check(rd('scenario/stsim_StockTypeGroupMembership_additions.csv').StockTypeId, stypes, 'Stock type')
fm = rd('scenario/stsim_FlowTypeGroupMembership_additions.csv')
check(fm.FlowTypeId, ftypes, 'Flow type')
problems += [f'Flow type {t!r} missing from SEEA_all_carbon_flows' for t in sorted(ftypes - set(fm[fm.FlowGroupId == 'SEEA_all_carbon_flows'].FlowTypeId))]
services = set(rd('project/stsimseea_Service.csv').Name)
check(rd('scenario/stsimseea_ServiceSupply.csv').ServiceId, services, 'Supply rule service')
check(rd('scenario/stsimseea_ServicePrice.csv').ServiceId, services, 'Price service')
print('\n'.join(problems) if problems else f'All names match: {len(classes)} classes, {len(ttypes)} transition types, '
      f'{len(stypes)} stock types, {len(ftypes)} flow types, {len(services)} services.')

In [ ]:
# 11. Put the SEEA-Sim tables into the library
!python {OUT}/add_seeasim.py --library {LIB} --scenario "{SCN}" --ssim /root/syncrosim
print(open(os.path.join(OUT, 'add_seeasim_log.txt'), encoding='utf-8').read())

In [ ]:
# 12. Read the library back: every SEEA-Sim table, the pipeline, the ST-Sim output switch
lib = ps.library(name=LIB, session=s)
proj = lib.projects(name='Definitions')
scn = proj.scenarios(name=SCN)
print('Packages in library:'); print(lib.packages)
for name in ['stsimseea_Service', 'stsim_StockGroup', 'stsim_FlowGroup']:
    d = proj.datasheets(name)
    print(f'\n{name} ({len(d)} rows)'); print(d.to_string(index=False))
for name in ['stsimseea_Options', 'stsimseea_AccountingPeriod', 'stsimseea_TransitionTypeClass',
             'stsimseea_CarbonStockGroup', 'stsimseea_CarbonFlowGroup', 'stsimseea_ServiceSupply',
             'stsimseea_ServicePrice', 'stsimseea_AssetValuation', 'stsim_StockTypeGroupMembership',
             'stsim_FlowTypeGroupMembership', 'core_Pipeline']:
    d = scn.datasheets(name)
    print(f'\n{name} ({len(d)} rows)'); print(d.to_string(index=False))
oo = scn.datasheets('stsim_OutputOptions')
print('\nTransitions by state class output:', oo[['SummaryOutputTRSC', 'SummaryOutputTRSCTimesteps']].to_string(index=False))

In [ ]:
# 13. Copy the library (and its .ssim.* folders), the logs and the package file to Drive
import datetime
out = os.path.join(DRIVE_DIR, 'colab_build_' + datetime.datetime.now().strftime('%Y%m%d_%H%M'))
os.makedirs(out, exist_ok=True)
for f in [LIB, os.path.join(WORK, 'build_log.txt'), os.path.join(OUT, 'add_seeasim_log.txt'), PKG_PATH]:
    if os.path.exists(f):
        shutil.copy2(f, out)
for d in glob.glob(LIB + '.*'):
    if os.path.isdir(d):
        shutil.copytree(d, os.path.join(out, os.path.basename(d)), dirs_exist_ok=True)
shutil.copytree(OUT, os.path.join(DRIVE_DIR, '4_seeasim'), dirs_exist_ok=True)
print('Saved to', out); print(sorted(os.listdir(out)))
print('\nOn the laptop: install', os.path.basename(PKG_PATH), 'in Studio, open JEM_cycle1.ssim, run 3 iterations first.')